# Enhancement

**Owner:** Ashok

Real-ESRGAN (Wang et al., 2021) upscales the whole photo 4x; GFPGAN (Wang et al., 2021) restores faces and runs only on detected faces. Both are pretrained and used for inference only.

| Item | Detail |
|---|---|
| Stage | Last stage of the pipeline (after repair and colorization) |
| Models | Real-ESRGAN `RealESRGAN_x4plus` for the whole image, GFPGAN v1.4 for faces |
| Input / output | RGB image -> RGB image, up to 4x larger, with restored faces |
| Skip rules | No face found -> GFPGAN skipped (output equals Real-ESRGAN alone); faces under 32 px skipped |
| Requirements | R2 (algorithm + theory), R4 (metrics), E1 (pretrained models) |

**Sections**
1. Setup
2. How the two models work (theory)
3. Load the models
4. Helpers: output-size cap, face filter
5. The `Enhancer` class (lives in `src/enhancement.py`)
6. Test images
7. Results and metrics (PSNR, SSIM, LPIPS)
8. Face blend weight (identity protection)
9. Tests
10. Resource smoke test
11. Findings and limits

> GFPGAN generates plausible facial detail from a learned face prior. It can change how a person looks, so the demo shows before and after, blends the restored face with the original at a set weight, and labels the output as AI-restored.

## 1. Setup

In [ ]:
import os, sys, subprocess
from pathlib import Path

# On Colab: clone the repo. Locally: walk up from the working directory to the project root.
REPO_URL = "https://github.com/diaeshantony-usd/old-photo-restoration-studio"
if Path("/content").exists() and not Path("/content/old-photo-restoration-studio").exists():
    subprocess.run(["git", "clone", REPO_URL, "/content/old-photo-restoration-studio"], check=True)

def get_project_root() -> Path:
    cur = Path.cwd().resolve()
    for c in [cur, *cur.parents, Path("/content/old-photo-restoration-studio")]:
        if (c / "src").exists() and (c / "notebooks").exists():
            return c
    return cur

ROOT = get_project_root()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
DATA_DIR = ROOT / "data" / "cleaned"          # produced by notebook 01 (or unzipped from the team Drive)

!pip install -q realesrgan gfpgan basicsr facexlib lpips scikit-image opencv-python-headless

# basicsr (a dependency of both models) still imports a torchvision module that newer
# torchvision removed. Point it at the module's new location.
import basicsr
deg = os.path.join(os.path.dirname(basicsr.__file__), "data", "degradations.py")
src = open(deg).read()
if "functional_tensor" in src:
    open(deg, "w").write(src.replace("torchvision.transforms.functional_tensor", "torchvision.transforms.functional"))
    print("patched", deg)
print("project root:", ROOT)

In [ ]:
import time, random
import numpy as np
import cv2
import torch
import pandas as pd
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
OUT_DIR = "notebooks/05_enhancement"       # small outputs the report needs are saved here
print("device:", DEVICE, "|", torch.cuda.get_device_name(0) if DEVICE == "cuda" else "no GPU (slow)")

## 2. How the two models work

**Real-ESRGAN** (whole image). A super-resolution model that turns a small, degraded image into a larger, sharper one. Its generator is an RRDB network (a deep stack of residual-in-residual dense blocks) trained against a discriminator, so textures look sharp instead of blurry. The key idea is **training on purely synthetic damage**: low-resolution inputs are made from clean images by applying blur, noise, resizing and JPEG compression *twice in a row* with random strengths ("high-order degradation"), plus ringing and overshoot artifacts. Because real old photos have mixed damage of this kind, the model generalizes to them without ever seeing a real one. This is the same synthetic-to-real argument our whole project makes.

**GFPGAN** (faces only). General upscalers invent smooth, generic faces. GFPGAN uses a **pretrained face GAN (StyleGAN2) as a prior**: a degradation-removal network reads the damaged face and predicts the latent codes and multi-scale features that steer the GAN to generate a clean face, in a **single forward pass**. Channel-split spatial feature transform layers keep the output faithful to the input where it is reliable, and a face-aware identity loss pushes the result toward the same person.

**How we combine them.** Real-ESRGAN upscales the whole photo. A face detector (RetinaFace, inside GFPGAN) finds faces; each one is aligned to a standard 512 x 512 crop, restored by GFPGAN, blended with the unrestored crop, and pasted back onto the Real-ESRGAN output. With no faces, only Real-ESRGAN runs.

Both models are inference only, so no training is needed and this stage fits a free T4.

## 3. Load the models

Weights are downloaded by `src/enhancement.py` from the official release pages of the two projects. E1 asks for pretrained models from Hugging Face; if the team wants to meet that literally, mirror these two files to a Hugging Face model repo in Phase 1 (this also covers the "downloads break" risk in the design doc) and change the two URLs.

In [ ]:
from src.enhancement import Enhancer, load_models, compute_outscale, keep_face, MAX_OUTPUT_SIDE, MIN_FACE_PX
from src.metrics import image_metrics, psnr, ssim, lpips_distance

upsampler, gfpgan = load_models(model_dir=str(ROOT / "models"))     # weights are downloaded, never committed
print("models loaded")

## 4. Helpers: output-size cap and face filter

Two rules from the design doc's corner-case table:

- **Very large image:** the upscale is capped so the longest output side is at most 4096 px.
- **Several faces of different sizes:** faces smaller than 32 px (on the input) are skipped, because there is too little detail to restore and the result would be invented.

In [ ]:
print(f"output cap: {MAX_OUTPUT_SIDE} px longest side; minimum face: {MIN_FACE_PX} px")
print("example: 3000x4000 input, 4x requested ->", compute_outscale(3000, 4000, 4.0), "x")

## 5. The `Enhancer` class

The class lives in `src/enhancement.py` so the notebook, the tests and the demo share one implementation. After each call, `last_info` records what happened: faces found, faces restored, faces skipped for size, the output scale used and the tile size.

Face handling uses GFPGAN's own face helper (detect, align, paste back) but with two changes: faces under 32 px are dropped before restoration, and each restored face is blended with the unrestored aligned face at `face_weight` (1.0 = fully restored, 0.0 = untouched).

In [ ]:
import logging
logging.basicConfig(level=logging.INFO, format="%(name)s: %(message)s")
enhancer = Enhancer(upsampler, gfpgan)

## 6. Test images

Real damaged scans have no high-resolution ground truth at the sizes we need, so to measure quality we do the reverse of enhancement: take a clean image as ground truth, make a degraded low-resolution copy, enhance it, and compare.

Two sets of clean images from `data/cleaned/` (made by notebook 01):

- **faces:** `humanface8000/color/`, the face-restoration test (the last N files in sorted order)
- **scenes:** `openphoto_restore/clean/`, the whole-image test (the last N files in sorted order)

`no_face_img` is a public photo of a coffee cup from scikit-image, used wherever a test needs an image that surely has no face. If `data/cleaned` is missing, the notebook falls back to scikit-image sample photos so every cell still runs.

Evaluation uses **2x** so the output size equals the ground-truth size. The visual demo uses the full 4x.

In [ ]:
from skimage import data

N_TEST, MAX_SIDE = 10, 512

def _load(folder, n):
    files = sorted(folder.glob("*.png"))[-n:] if folder.exists() else []
    out = {}
    for f in files:
        im = cv2.cvtColor(cv2.imread(str(f)), cv2.COLOR_BGR2RGB)
        out[f.stem] = im
    return out

def _fit(im, side=MAX_SIDE):
    sc = side / max(im.shape[:2])
    im = cv2.resize(im, (round(im.shape[1] * sc), round(im.shape[0] * sc)), interpolation=cv2.INTER_AREA)
    return im[: im.shape[0] // 2 * 2, : im.shape[1] // 2 * 2]           # even size so 2x down/up matches

def degrade(gt: np.ndarray, scale: int = 2, seed: int = SEED) -> np.ndarray:
    """Make a realistic low-res copy: blur, downsample, noise, JPEG."""
    rng = np.random.default_rng(seed)
    x = cv2.GaussianBlur(gt, (0, 0), 1.0)
    x = cv2.resize(x, (gt.shape[1] // scale, gt.shape[0] // scale), interpolation=cv2.INTER_AREA)
    x = np.clip(x.astype(np.float32) + rng.normal(0, 3, x.shape), 0, 255).astype(np.uint8)
    ok, enc = cv2.imencode(".jpg", cv2.cvtColor(x, cv2.COLOR_RGB2BGR), [cv2.IMWRITE_JPEG_QUALITY, 60])
    return cv2.cvtColor(cv2.imdecode(enc, 1), cv2.COLOR_BGR2RGB)

faces = {k: _fit(v) for k, v in _load(DATA_DIR / "humanface8000" / "color", N_TEST).items()}
scenes = {k: _fit(v) for k, v in _load(DATA_DIR / "openphoto_restore" / "clean", N_TEST).items()}
if not faces:
    print("data/cleaned not found: using scikit-image fallback photos")
    faces = {"astronaut": _fit(data.astronaut())}
    scenes = {"chelsea": _fit(data.chelsea()), "rocket": _fit(data.rocket())}
no_face_img = _fit(data.coffee())
face_img = next(iter(faces.values()))
samples = {**scenes, **faces}
face_names = set(faces)
print(f"{len(scenes)} scenes, {len(faces)} faces")

show = list(samples.items())[:6]
fig, axs = plt.subplots(1, len(show), figsize=(3.2 * len(show), 3))
for ax, (n, im) in zip(np.atleast_1d(axs), show):
    ax.imshow(im); ax.set_title(n[:18], fontsize=8); ax.axis("off")
plt.show()

## 7. Results and metrics

| Metric | Measures | Better |
|---|---|---|
| PSNR (dB) | Pixel-level closeness to the ground truth | Higher |
| SSIM (0-1) | Whether edges and textures match (Wang et al., 2004) | Higher |
| LPIPS | How different it looks to people, from deep features (Zhang et al., 2018) | Lower |

Three methods are compared on the same degraded inputs: **bicubic** resizing (the classical baseline), **Real-ESRGAN only**, and **Real-ESRGAN + GFPGAN**.

**Expect PSNR to disagree with LPIPS.** GAN upscalers add sharp, plausible texture that is not pixel-identical to the original, so bicubic can win on PSNR while Real-ESRGAN wins clearly on LPIPS and in how the image looks. We report all three and say so, rather than pick the one that flatters the model.

In [ ]:
metrics = image_metrics          # PSNR, SSIM, LPIPS from src/metrics.py

EVAL_SCALE = 2
no_face = Enhancer(upsampler, gfpgan, scale=EVAL_SCALE, restore_faces=False)
with_face = Enhancer(upsampler, gfpgan, scale=EVAL_SCALE, restore_faces=True)

rows, panels = [], []
for name, gt in samples.items():
    lr = degrade(gt, EVAL_SCALE)
    outs = {
        "bicubic": cv2.resize(lr, (gt.shape[1], gt.shape[0]), interpolation=cv2.INTER_CUBIC),
        "Real-ESRGAN": no_face.enhance(lr),
        "Real-ESRGAN + GFPGAN": with_face.enhance(lr),
    }
    faces = with_face.last_info["faces_restored"]
    for method, out in outs.items():
        rows.append(dict(image=name, kind='face' if name in face_names else 'scene', method=method, faces_restored=faces if "GFPGAN" in method else 0,
                         **metrics(out, gt)))
    panels.append((name, lr, outs, gt))

df = pd.DataFrame(rows)
display(df.round(3))
print("mean over images:")
display(df.groupby(["kind", "method"])[["psnr", "ssim", "lpips"]].mean().round(3))

In [ ]:
fig, axs = plt.subplots(len(panels), 5, figsize=(18, 3.6 * len(panels)))
for row, (name, lr, outs, gt) in zip(axs, panels):
    ims = [("degraded input", cv2.resize(lr, (gt.shape[1], gt.shape[0]), interpolation=cv2.INTER_NEAREST))] + \
          list(outs.items()) + [("ground truth", gt)]
    for ax, (t, im) in zip(row, ims):
        ax.imshow(im); ax.set_title(f"{name}\n{t}", fontsize=8); ax.axis("off")
plt.tight_layout()
fig.savefig(f"{OUT_DIR}/enhancement_grid.png", dpi=110)
df.to_csv(f"{OUT_DIR}/enhancement_metrics.csv", index=False)
plt.show()

## 8. Face blend weight (identity protection)

`face_weight` blends each restored face with the unrestored one. Higher is sharper but riskier for identity; lower keeps more of the original. This sweep on a real face image shows the trade-off so the team can pick a default and justify it. The face crop is shown at 4x.

In [ ]:
gt = face_img
lr = degrade(gt, EVAL_SCALE)
sweep, crops = [], []
for w in (0.0, 0.3, 0.5, 0.7, 1.0):
    e = Enhancer(upsampler, gfpgan, scale=EVAL_SCALE, face_weight=w)
    out = e.enhance(lr)
    sweep.append(dict(face_weight=w, **metrics(out, gt)))
    crops.append((w, out))
display(pd.DataFrame(sweep).round(3))

fig, axs = plt.subplots(1, len(crops), figsize=(3.2 * len(crops), 3.6))
for ax, (w, out) in zip(axs, crops):
    H, W = out.shape[:2]; ax.imshow(out[H // 5: 4 * H // 5, W // 4: 3 * W // 4]); ax.set_title(f"weight {w}"); ax.axis("off")   # centre crop: portraits are face-centred
plt.tight_layout()
fig.savefig(f"{OUT_DIR}/face_weight_sweep.png", dpi=110)
plt.show()

## 9. Tests

The output-cap and face-size tests that need no model already run as pytest cases in `tests/test_enhancement_helpers.py`; these cells run the model-based checks.

In [ ]:
def test_output_size_is_scaled_input():
    lr = degrade(no_face_img, 2)
    out = Enhancer(upsampler, gfpgan, scale=4.0).enhance(lr)
    assert out.shape[:2] == (lr.shape[0] * 4, lr.shape[1] * 4) and out.dtype == np.uint8

def test_no_face_means_no_gfpgan_and_equals_real_esrgan_alone():
    lr = degrade(no_face_img, 2)
    full = Enhancer(upsampler, gfpgan, scale=2.0)
    alone = Enhancer(upsampler, gfpgan, scale=2.0, restore_faces=False)
    a, b = full.enhance(lr), alone.enhance(lr)
    assert full.last_info["faces_found"] == 0 and "no face" in full.last_info["face_step"]
    assert np.array_equal(a, b)

def test_face_is_found_and_restored():
    e = Enhancer(upsampler, gfpgan, scale=2.0)
    e.enhance(degrade(face_img, 2))
    assert e.last_info["faces_restored"] >= 1

def test_tiny_faces_are_skipped():
    tiny = cv2.resize(degrade(face_img, 2), (64, 64))     # face far under 32 px
    e = Enhancer(upsampler, gfpgan, scale=2.0)
    e.enhance(tiny)
    assert e.last_info["faces_restored"] == 0

def test_switched_off_returns_input():
    lr = degrade(no_face_img, 2)
    e = Enhancer(upsampler, gfpgan, enabled=False)
    assert np.array_equal(e.enhance(lr), lr) and e.last_info["ran"] is False

def test_output_cap():
    assert compute_outscale(1000, 1000, 4.0) == 4.0
    assert compute_outscale(3000, 4000, 4.0) == 1.024           # 4000 * 1.024 = 4096
    assert compute_outscale(8000, 6000, 4.0) == 1.0             # never below 1x

def test_face_size_filter():
    assert keep_face([0, 0, 40, 50, 0.9]) and not keep_face([0, 0, 20, 50, 0.9]) and not keep_face([0, 0, 50, 31, 0.9])

def test_same_input_same_output():
    lr = degrade(no_face_img, 2)
    e = Enhancer(upsampler, gfpgan, scale=2.0)
    assert np.array_equal(e.enhance(lr), e.enhance(lr))

LARGE_TEST = False     # set True on a T4 to run the 4000 x 3000 case from the design doc (slow)
def test_large_image():
    big = cv2.resize(no_face_img, (4000, 3000))
    e = Enhancer(upsampler, gfpgan)
    out = e.enhance(big)
    assert max(out.shape[:2]) <= MAX_OUTPUT_SIDE

tests = [v for k, v in sorted(globals().items()) if k.startswith("test_") and (k != "test_large_image" or LARGE_TEST)]
for t in tests:
    t(); print("PASS", t.__name__)

## 10. Resource smoke test

Logs peak GPU memory and seconds per image so the numbers can replace the plan in the design doc's GPU budget table (section 3.4). Each size is run with and without a face in the image.

In [ ]:
def smoke_test(sizes=((256, 256), (512, 512), (1024, 768)), repeats=2):
    rows = []
    e = Enhancer(upsampler, gfpgan, scale=4.0)
    for (w, h) in sizes:
        for label, base in (("face", face_img), ("no face", no_face_img)):
            src = cv2.resize(base, (w, h))
            e.enhance(src)                                    # warm-up
            if DEVICE == "cuda":
                torch.cuda.reset_peak_memory_stats(); torch.cuda.synchronize()
            t0 = time.time()
            for _ in range(repeats):
                e.enhance(src)
            if DEVICE == "cuda":
                torch.cuda.synchronize()
            rows.append(dict(input=f"{w}x{h}", content=label, output=f"{w*4}x{h*4}",
                             sec_per_image=(time.time() - t0) / repeats,
                             peak_gpu_gb=(torch.cuda.max_memory_allocated() / 1e9) if DEVICE == "cuda" else float("nan")))
    return pd.DataFrame(rows)

smoke = smoke_test()
display(smoke.round(3))
smoke.to_csv(f"{OUT_DIR}/enhancement_smoke_test.csv", index=False)

## 11. Findings and limits

*Fill in after running on the Colab T4 and on the OpenPhoto Restore and film-scan images.*

- PSNR, SSIM, LPIPS: bicubic vs Real-ESRGAN vs Real-ESRGAN + GFPGAN: _TBD_
- Chosen default `face_weight` and why: _TBD_
- Peak memory and seconds per image on a T4 (and whether the 2x fallback is needed): _TBD_
- Failure cases (identity drift, over-smoothed texture, faces in profile or partly hidden, halo at tile seams): _TBD_
- Order in the pipeline: enhancement runs last so it upscales the repaired, colorized photo; check that it does not amplify leftover damage that repair missed: _TBD_

**Known limits.** Enhancement adds plausible detail, it does not recover true detail. GFPGAN is trained on faces and can hallucinate features on very damaged ones, hence the blend weight and the before/after display. The 4096 px output cap means very large scans are upscaled less than 4x.

**Code location:** `compute_outscale`, `keep_face`, `Enhancer`, `load_models` and the constants live in `src/enhancement.py`; metrics come from `src/metrics.py`. `app/pipeline.py` calls the same classes.

## References

- Wang, X., Xie, L., Dong, C., & Shan, Y. (2021). Real-ESRGAN: Training real-world blind super-resolution with pure synthetic data. *ICCVW 2021*.
- Wang, X., Li, Y., Zhang, H., & Shan, Y. (2021). Towards real-world blind face restoration with generative facial prior. *CVPR 2021*.
- Wang, Z., Bovik, A. C., Sheikh, H. R., & Simoncelli, E. P. (2004). Image quality assessment: From error visibility to structural similarity. *IEEE Transactions on Image Processing*.
- Zhang, R., Isola, P., Efros, A. A., Shechtman, E., & Wang, O. (2018). The unreasonable effectiveness of deep features as a perceptual metric. *CVPR 2018*.